In [1]:
import pyNUISANCE as pn

In [2]:
fin = "/root/Projects/pico/NOEC/evs/bla.hepmc3"

evs = pn.EventSource(fin)
if not evs:
    print("Failed to read event file")

In [3]:
from pyProSelecta import event, part, unit, pdg, p3mod, momentum, kinetic_energy, ext
from pyNUISANCE import nhm
import numpy as np

def GetPrimaryLepton(ev):
    beam_part = nhm.EventUtils.GetBeamParticle(ev)
    prim_vertex = nhm.EventUtils.GetPrimaryVertex(ev)
    for part in prim_vertex.particles_out():
        if (part.pid() == beam_part.pid()) or\
           ( (abs(part.pid()) + 1 ) == abs(beam_part.pid())):
          return part
    raise runtime_error("Failed to find primary lepton out: %s -> [%s]" % \
                        (beam_part.pid(), ",".join(part.pid for part in prim_vertex.particles_out())))

def PrimaryLeptonTotalEnergy(ev):
    pl = GetPrimaryLepton(ev)
    return (pl.momentum().e() / unit.GeV_c) if pl.pid() % 2 else 0

def NeutralPionTotalEnergy(ev):
    return part.sum(momentum, event.all_out_part(ev, 111)).e() / unit.GeV_c

def NumberOfChargedPions(ev):
    return event.num_out_part(ev, [211,-211], flatten=True)

def ChargedPionKineticEnergy(ev):
    return part.sum(kinetic_energy, event.all_out_part(ev, [211, -211], flatten=True)) / unit.GeV_c

def ProtonKineticEnergy(ev):
    return part.sum(kinetic_energy, event.all_out_part(ev, 2212)) / unit.GeV_c

def ReconstructedNeutrinoEnergy_true(ev):
    return PrimaryLeptonTotalEnergy(ev) + ChargedPionKineticEnergy(ev) + (NumberOfChargedPions(ev)*0.1395) +  NeutralPionTotalEnergy(ev) + ProtonKineticEnergy(ev)

efg = pn.EventFrameGen(evs) \
        .add_column("pid_nu", lambda ev: nhm.EventUtils.GetBeamParticle(ev).pid()) \
        .add_column("E_nu", ext.enu_GeV) \
        .add_column("is_CC", ext.isCC) \
        .add_column("pid_lep", lambda ev: GetPrimaryLepton(ev).pid()) \
        .add_column("E_lepton", PrimaryLeptonTotalEnergy) \
        .add_column("T_proton", ProtonKineticEnergy) \
        .add_column("num_pi0", lambda ev: event.num_out_part(ev, 111)) \
        .add_column("E_pi0", NeutralPionTotalEnergy) \
        .add_column("num_cpi", NumberOfChargedPions) \
        .add_column("T_cpi", ChargedPionKineticEnergy) \
        .add_column("E_nu_rec_true", ReconstructedNeutrinoEnergy_true)

print(efg.first(20))

output_columns = [
    "pid_nu",
    "E_nu",
    "is_CC",
    "pid_lep",
    "E_lepton",
    "T_proton",
    "num_pi0",
    "E_pi0",
    "num_cpi",
    "T_cpi",
    "E_nu_rec_true" ]

 -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 | event.number | weight.cv | fatx_per_su$ | fatx_per_su$ | process.id | pid_nu |   E_nu | is_CC | pid_lep | E_lepton | T_proton | num_pi0 | E_pi0 | num_cpi |  T_cpi | E_nu_rec_tr$ |
 -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 |            0 |         1 |    2.491e+14 |    1.557e+13 |        200 |     14 | 0.5859 |     1 |      13 |   0.5067 |  0.06351 |       0 |     0 |       0 |      0 |       0.5702 |
 |            1 |         1 |    1.246e+14 |    7.785e+12 |        200 |     14 | 0.3957 |     1 |      13 |   0.2562 |  0.08987 |       0 |     0 |       0 |      0 |       0.3461 |
 |            2 |         1 |    8.304e+13 |     5.19e+12 |        200 |     14 | 0.3

In [4]:
ef = efg.all()
print(ef.table.shape[0])

10000


In [5]:
import pandas as pa

def todf(ef):
  df = pa.DataFrame(data=ef.table,
                    index=[i for i in range(ef.table.shape[0])],
                    columns=ef.column_names)
  for row in ["event.number", "process.id", "pid_nu", "is_CC", "pid_lep", "num_pi0", "num_cpi"]:
    df[row] = pa.to_numeric(df[row], downcast="integer")
  return df
df = todf(ef)
df

,event.number,weight.cv,fatx_per_sumw.pb_per_target.estimate,fatx_per_sumw.pb_per_nucleon.estimate,process.id,pid_nu,E_nu,is_CC,pid_lep,E_lepton,T_proton,num_pi0,E_pi0,num_cpi,T_cpi,E_nu_rec_true
0,0,1.0,2.491137e+14,1.556961e+13,200,14,0.585928,1,13,0.506730,0.063506,0,0.000000,0,0.000000,0.570236
1,1,1.0,1.245569e+14,7.784804e+12,200,14,0.395725,1,13,0.256209,0.089869,0,0.000000,0,0.000000,0.346078
2,2,1.0,8.303791e+13,5.189869e+12,200,14,0.396580,1,13,0.215562,0.162075,0,0.000000,0,0.000000,0.377638
3,3,1.0,6.227843e+13,3.892402e+12,400,14,0.581634,1,13,0.204422,0.031275,0,0.000000,1,0.192167,0.567363
4,4,1.0,4.982274e+13,3.113921e+12,200,14,0.853698,1,13,0.409232,0.421384,0,0.000000,0,0.000000,0.830617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9995,1.0,2.647892e+10,1.557584e+09,200,14,0.774970,1,13,0.716482,0.033368,0,0.000000,0,0.000000,0.749850
9996,9996,1.0,2.647628e+10,1.557428e+09,251,14,0.793409,0,14,0.000000,0.000000,0,0.000000,0,0.000000,0.000000
9997,9997,1.0,2.647363e+10,1.557272e+09,200,14,0.758224,1,13,0.239841,0.532684,0,0.000000,0,0.000000,0.772525
9998,9998,1.0,2.647098e+10,1.557116e+09,451,14,0.702338,0,14,0.000000,0.182191,1,0.202631,0,0.000000,0.384822


In [6]:
df.to_csv("neut_test.csv",index_label="event_num",columns=output_columns, float_format="%.4f")